# 00 — Exploração dos dados negócio a negócio: WDO e WIN

**Objetivo:** antes de aplicar qualquer método do curso, entender os dados — o que é um negócio, como o volume se distribui no dia,
como são os retornos (caudas, autocorrelação) e se o fluxo de agressão tem alguma relação com o preço no minuto seguinte.

Conceitos do curso usados aqui: **Cap. 2** (o que é $Y = f(X) + \varepsilon$, predição vs. inferência).

**Pré-requisito:** arquivos da B3 em `data/raw/` (ver README, seção *Dados*).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.b3_loader import load_raw_dir
from src.features import front_month, time_bars, add_forward_return

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.figsize"] = (11, 4)

## 1. Carregar e salvar em Parquet
Ler os `.zip` brutos é lento; o Parquet fica pronto para os próximos notebooks.

In [ ]:
trades = load_raw_dir(RAW)
trades.to_parquet(PROCESSED / "trades_wdo_win.parquet")
print(f"{len(trades):,} negócios | {trades.ts.dt.date.nunique()} pregões")
trades.groupby([trades.ts.dt.date.rename("dia"), "ticker"]).agg(negocios=("price", "size"), contratos=("qty", "sum")).unstack()

## 2. Vencimento principal e barras de 1 minuto
Cada dia tem vários vencimentos negociando; ficamos com o de maior volume.

In [ ]:
fm = front_month(trades)
bars = {root: add_forward_return(time_bars(fm[fm.root == root], "1min"), 1) for root in ["WDO", "WIN"]}
bars["WDO"].head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, (root, b) in zip(axes, bars.items()):
    ax.plot(range(len(b)), b["close"].values, lw=0.8)
    ax.set_title(f"{root} — fechamento de 1 min (eixo x = barras, sem buracos de noite)")
plt.tight_layout()

## 3. Perfil intradiário do volume
Espera-se um formato de "U": mais volume na abertura e no fechamento.

In [ ]:
fig, ax = plt.subplots()
for root, b in bars.items():
    prof = b.groupby(b.index.strftime("%H:%M"))["volume"].mean()
    ax.plot(prof.index, prof / prof.mean(), label=root)
ax.set_xticks(ax.get_xticks()[::60]); ax.set_ylabel("volume / média"); ax.legend(); ax.set_title("Volume médio por minuto do dia")

## 4. Distribuição dos retornos de 1 minuto
Retornos financeiros têm **caudas gordas**: curtose bem acima de 3 (a da normal).

In [ ]:
fig, axes = plt.subplots(1, 2)
for ax, (root, b) in zip(axes, bars.items()):
    r = b["ret"].dropna()
    ax.hist(r / r.std(), bins=100, density=True, alpha=0.7)
    x = np.linspace(-5, 5, 200); ax.plot(x, np.exp(-x**2 / 2) / np.sqrt(2 * np.pi), "k--", lw=1)
    ax.set_xlim(-6, 6); ax.set_title(f"{root}: curtose = {r.kurt() + 3:.1f}")
plt.tight_layout()

## 5. Autocorrelação
Autocorrelação negativa no lag 1 costuma vir do *bid-ask bounce* (o preço alterna entre compra e venda).

In [ ]:
fig, axes = plt.subplots(1, 2)
for ax, (root, b) in zip(axes, bars.items()):
    plot_acf(b["ret"].dropna(), lags=20, ax=ax, title=f"ACF retornos 1 min — {root}", zero=False)
plt.tight_layout()

## 6. Fluxo de agressão → retorno seguinte (prévia do Cap. 3)
`imbalance` = (volume comprador − vendedor) / volume, estimado pela *tick rule* (o arquivo público não traz o agressor).

Atenção à diferença: a correlação **no mesmo minuto** é quase mecânica (a tick rule usa o próprio preço);
o que interessa para predição é a correlação com o **minuto seguinte** (`fwd_ret_1`).

In [ ]:
for root, b in bars.items():
    d = b[["imbalance", "ret", "fwd_ret_1"]].dropna()
    print(f"{root}: corr(imbalance, ret mesmo minuto) = {d.imbalance.corr(d.ret):+.3f} | "
          f"corr(imbalance, ret próximo minuto) = {d.imbalance.corr(d.fwd_ret_1):+.3f} | n = {len(d):,}")

## 7. WDO × WIN
Dólar e bolsa costumam andar em sentidos opostos (fuga de risco → dólar sobe, índice cai).

In [ ]:
both = pd.concat({k: v["ret"] for k, v in bars.items()}, axis=1).dropna()
print(f"corr(ret WDO, ret WIN) em 1 min = {both.corr().iloc[0, 1]:+.3f}")
both.plot.scatter("WIN", "WDO", s=3, alpha=0.3, title="Retornos de 1 min");

## Conclusões e próximos passos
*(preencher após rodar com os dados reais)*

- …
- Próximo: `ch03_regressao_linear.ipynb` — regressão de `fwd_ret` em `imbalance`, retornos defasados e volume, com diagnósticos do Cap. 3.